In [0]:
# Add widgets (parameters)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text(
    "bronze_path",
    "/Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events"
)
dbutils.widgets.text(
    "silver_path",
    "/Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events_v2"
)

bronze_path = dbutils.widgets.get("bronze_path")
silver_path = dbutils.widgets.get("silver_path")

print("bronze_path =", bronze_path)
print("silver_path =", silver_path)

if bronze_path.strip() == silver_path.strip():
    raise Exception("Config error: silver_path cannot be the same as bronze_path")

In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

silver_typed = (
    bronze_df
    .withColumn("event_ts", F.to_timestamp("event_time"))
    .drop("event_time")
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("event_date", F.to_date("event_ts"))
)

silver_valid = (
    silver_typed
    .filter(F.col("event_ts").isNotNull())
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("price").isNotNull())
    .filter((F.col("price") > 0) & (F.col("price") < 10000))
)

cols = set(silver_valid.columns)
if {"user_session", "event_ts"}.issubset(cols):
    keys = ["user_session", "event_ts"]
elif {"user_id", "product_id", "event_ts"}.issubset(cols):
    keys = ["user_id", "product_id", "event_ts"]
else:
    raise Exception(f"No suitable dedupe keys found. Columns: {sorted(cols)}")

# Window for deterministic keep-latest (by ingestion_ts)
w = Window.partitionBy(*keys).orderBy(F.col("ingestion_ts").desc())

silver = (
    silver_valid
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn(
        "price_tier",
        F.when(F.col("price") < 10, "budget")
         .when(F.col("price") < 50, "mid")
         .otherwise("premium")
    )
)

# Guardrails: prevent layer corruption (common orchestration mistake)
if "bronze_path" in globals():
    if silver_path.strip() == bronze_path.strip():
        raise Exception(f"Config error: silver_path equals bronze_path. Refusing to overwrite Bronze. path={silver_path}")

if "/delta/bronze/" in silver_path.replace("\\", "/"):
    raise Exception(f"Safety check failed: silver_path points to Bronze. Refusing to write. path={silver_path}")

# Enforce a stable Silver contract (prevents accidental schema drift)
expected_cols = [
    "event_type", "product_id", "category_id", "category_code", "brand", "price",
    "user_id", "user_session", "batch_id", "ingestion_ts",
    "event_ts", "event_date", "price_tier"
]
silver = silver.select(*[c for c in expected_cols if c in silver.columns])

# Materialize metrics before write (cleaner job logs, avoids surprise recomputes)
silver_rows = silver.count()

# Write Silver
silver.write.format("delta").mode("overwrite").save(silver_path)

# Exit message used by Jobs UI
dbutils.notebook.exit(f"OK silver rows={silver_rows} keys={keys} output={silver_path}")